In [0]:
# Define paths
bronze_path = "/Volumes/workspace/default/olist_raw/bronze/"
silver_path = "/Volumes/workspace/default/olist_raw/silver/"

# Load bronze tables
orders = spark.read.format("delta").load(bronze_path + "orders")
customers = spark.read.format("delta").load(bronze_path + "customers")
order_items = spark.read.format("delta").load(bronze_path + "order_items")
payments = spark.read.format("delta").load(bronze_path + "payments")
reviews = spark.read.format("delta").load(bronze_path + "reviews")
products = spark.read.format("delta").load(bronze_path + "products")
sellers = spark.read.format("delta").load(bronze_path + "sellers")
category_translation = spark.read.format("delta").load(bronze_path + "category_translation")

print("Bronze tables loaded successfully")

Bronze tables loaded successfully


In [0]:
from pyspark.sql.functions import col, to_timestamp, datediff, when, isnan

# Clean orders table
orders_clean = orders \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .withColumn("delivery_days", datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )) \
    .withColumn("delivery_on_time", when(
        col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .dropDuplicates(["order_id"]) \
    .filter(col("order_status") == "delivered")

print(f"Orders after cleaning: {orders_clean.count()} rows")

Orders after cleaning: 96478 rows


In [0]:
from pyspark.sql.functions import round

# Clean order items and join with products and category translation
order_items_clean = order_items \
    .join(products, "product_id", "left") \
    .join(category_translation, "product_category_name", "left") \
    .withColumn("price", round(col("price"), 2)) \
    .withColumn("freight_value", round(col("freight_value"), 2)) \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .drop("product_category_name") \
    .withColumnRenamed("product_category_name_english", "product_category")

print(f"Order items after cleaning: {order_items_clean.count()} rows")
order_items_clean.show(3)

Order items after cleaning: 112650 rows
+--------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------+
|          product_id|            order_id|order_item_id|           seller_id|shipping_limit_date|price|freight_value|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|    product_category|
+--------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------+
|5a419dbf24a8c9718...|0010b2e5201cc5f1a...|            1|3504c0cb71d7fa48d...|2017-09-15 18:04:37| 48.9|         16.6|              

In [0]:
from pyspark.sql.functions import length

# Clean reviews
reviews_clean = reviews \
    .dropDuplicates(["review_id"]) \
    .withColumn("review_comment_length", length(col("review_comment_message"))) \
    .select(
        "order_id",
        "review_score",
        "review_comment_message",
        "review_comment_length"
    )

# Join everything into one master silver table
silver_master = orders_clean \
    .join(customers, "customer_id", "left") \
    .join(order_items_clean, "order_id", "left") \
    .join(payments, "order_id", "left") \
    .join(reviews_clean, "order_id", "left") \
    .join(sellers, "seller_id", "left")

# Write to silver layer
silver_master.write.format("delta").mode("overwrite").save(silver_path + "master")

print(f"Silver master table: {silver_master.count()} rows, {len(silver_master.columns)} columns")


Silver master table: 115361 rows, 38 columns
